In [ ]:
# import sys
# import time
# import clr

# sys.path.append(r"C:\Program Files\Thorlabs\Kinesis")
# clr.AddReference("Thorlabs.MotionControl.DeviceManagerCLI")
# clr.AddReference("Thorlabs.MotionControl.FilterFlipperCLI") 
# clr.AddReference("Thorlabs.MotionControl.KCube.DCServoCLI")
# clr.AddReference("System")

# from Thorlabs.MotionControl.DeviceManagerCLI import DeviceManagerCLI
# from Thorlabs.MotionControl.FilterFlipperCLI import FilterFlipper
# from Thorlabs.MotionControl.KCube.DCServoCLI import KCubeDCServo
# from System import Decimal
# from Thorlabs.MotionControl.GenericMotorCLI.Settings import RotationSettings



In [ ]:

# wait4action = 0.5
# rotation_serial_num = "27600279"


# print("Building device list for filter...")
# DeviceManagerCLI.BuildDeviceList()


# print(f"Connecting to Filter Rotation Stage (Serial: {rotation_serial_num})...")
# rotation = KCubeDCServo.CreateKCubeDCServo(rotation_serial_num)
# rotation.Connect(rotation_serial_num)

# rotation.LoadMotorConfiguration(rotation_serial_num)
# rotation.StartPolling(250)
# time.sleep(wait4action)
# print("Done initializing filter!")

Building device list for filter...
Connecting to Filter Rotation Stage (Serial: 27600279)...
Done initializing filter!


In [ ]:

# rotation.MotorDeviceSettings.Rotation.RotationDirection = RotationSettings.RotationDirections.Quickest
# rotation.SetSettings(rotation.MotorDeviceSettings, False, False)


In [ ]:
# current_mode = rotation.MotorDeviceSettings.Rotation.RotationDirection
# # 2. Print the result
# print(f"Current Rotation Mode: {current_mode}")

Current Rotation Mode: Forwards


In [3]:
import filter
filter.filter_init()
filter.filter_on()  
filter.flip_up()

Building device list for filter...
Connecting to Filter Flipper (Serial: 37010764)...
Connecting to Filter Rotation Stage (Serial: 27600279)...
Initializing filter...
Done initializing filter!
Turning filter on...
Done turning on!
Flipping up...
Done Flipping!


In [9]:
filter.rotation_home()

Homing rotation stage...
Done Homing!


In [2]:
filter.rotation_move(355)

Moving to 355 degrees...
Done Moving!


In [8]:

filter.rotation_move(20)

Moving to 20 degrees...
Done Moving!


In [ ]:
filter.set_rotation_mode("quickest")

In [ ]:

# Safely turn off the hardware
filter.filter_off()
print("Hardware disconnected. Calibration complete!")

In [2]:
import numpy as np

import os

def generate_corrected_calibration(folder_path, threshold=10000, wl_min=520, wl_max=660):
    """
    Reads raw calibration data from a directory, calculates the center wavelength 
    using the edge-threshold method (midpoint), and saves a corrected table.
    
    Args:
        folder_path (str): The path to the directory containing the .npy files.
        threshold (int): The intensity threshold to define the rising/falling edges.
        wl_min (int): The minimum wavelength of the search window.
        wl_max (int): The maximum wavelength of the search window.
    """
    # Define file paths
    wl_path = os.path.join(folder_path, 'wavelength.npy')
    table_path = os.path.join(folder_path, 'calibration_table.npy')
    data_path = os.path.join(folder_path, 'calibration_data.npy')
    output_path = os.path.join(folder_path, 'calibration_table_corrected.npy')

    # Load the raw data
    try:
        wl = np.load(wl_path)
        angles_raw = np.load(table_path)[:, 0] # Extract just the angles
        data_raw = np.load(data_path)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return None

    # Restrict the search to the specified wavelength window
    window_mask = (wl >= wl_min) & (wl <= wl_max)
    wl_window = wl[window_mask]

    corrected_table = []

    # Process each spectrum in the dataset
    for i in range(len(angles_raw)):
        angle = angles_raw[i]
        spectrum = data_raw[i, window_mask]
        
        # Find all indices where the signal is above the threshold
        above_thresh = np.where(spectrum > threshold)[0]
        
        if len(above_thresh) > 0:
            # The first time it crosses the threshold
            rise_idx = above_thresh[0]
            # The last time it is above the threshold
            fall_idx = above_thresh[-1]
            
            rise_wl = wl_window[rise_idx]
            fall_wl = wl_window[fall_idx]
            
            # Calculate the mathematical center between the two edges
            center_wl = (rise_wl + fall_wl) / 2.0
        else:
            # If the peak never hits the threshold at this specific angle
            center_wl = np.nan
            
        corrected_table.append([angle, center_wl])

    # Convert to a numpy array and save it
    corrected_table_arr = np.array(corrected_table)
    np.save(output_path, corrected_table_arr)
    
    print(f"Corrected calibration table successfully saved to:\n{output_path}")
    return output_path

save_dir = "calibration/2026-04-07_14-32-53"
print("\nGenerating corrected calibration table based on 10k threshold...")
generate_corrected_calibration(save_dir)


Generating corrected calibration table based on 10k threshold...
Corrected calibration table successfully saved to:
calibration/2026-04-07_14-32-53\calibration_table_corrected.npy


'calibration/2026-04-07_14-32-53\\calibration_table_corrected.npy'